[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skous2/is455-w2027/blob/main/module_05/Worksheet_5A_EDA.ipynb)

# Worksheet 5A · Exploratory data analysis

**Optional and not graded.** Work through it before Tuesday's class (Feb 9). Tuesday's walk-through covers sections 1–4 on one column at a time, and Thursday's class covers sections 5–9 on how each column relates to churn. Expect about 40–50 minutes.

**Before you start,** choose **File → Save a copy in Drive** so your work is saved. Try each problem yourself first. When you get stuck, ask your AI assistant for hints instead of the answer, the way the tutor prompt in Modules 1 and 2 did. Write answers in your own words.

Each section starts with a short worked example on made-up smoothie-shop data, then gives you the same kind of problem for Canyon Fitness. Each section also names the part of [OpenIntro Statistics](https://www.openintro.org/book/os/) that covers it. The PDF is free if you set the price to $0.

## The question this week

**Exploratory data analysis (EDA)** is the data understanding phase of CRISP-DM. Before you build a model, you look at each column on its own, then at how each one relates to the outcome. EDA tells you which features are worth trying, which values need attention, and which columns would leak the answer.

The outcome this week is the churn definition Module 3 chose for monthly retention offers. A member has churned if they have no visit in the 30 days after October 1.

## 1. Rows, the target and the baseline

*[OpenIntro Statistics](https://www.openintro.org/book/os/), section 1.2, "Data basics," defines observations, variables and their types.*

**Worked example.** The made-up table below has one row per customer. The column you want to predict is called the **target**, here `lapsed`. The target's rate gives you a **baseline**. Guessing the most common outcome for everyone is already right that often, so a model has to beat it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Made-up data for 500 smoothie-shop customers
rng = np.random.default_rng(455)
shop = pd.DataFrame({
    "spend_last_month":  rng.gamma(1.5, 12, 500).round(2),
    "visits_last_month": rng.poisson(4, 500),
    "member":            rng.random(500) < 0.4,
    "favorite":          rng.choice(["Mango", "Berry", "Green", "Seasonal"], 500, p=[0.5, 0.3, 0.17, 0.03]),
})
shop["lapsed"] = rng.random(500) < np.where(shop["visits_last_month"] < 2, 0.55, 0.15)

print(shop.shape)
print(shop["lapsed"].mean())          # share of customers who lapsed
print(1 - shop["lapsed"].mean())      # accuracy of guessing "stays" for everyone

**Your turn.**

Run the cell below to load the data.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/skous2/is455-w2027/main/data/canyon_fitness_members.csv"
members = pd.read_csv(DATA_URL, parse_dates=["join_date", "next_visit_date", "cancel_date"])
snapshot = pd.Timestamp("2026-10-01")
print(members.shape)

**1a.** Add a `churned` column that is `True` when `next_visit_date` is missing or more than 30 days after `snapshot`. Print the churn rate, and print the accuracy of guessing that every member stays.

In [ ]:
# Your code here


**1b.** Which four columns should EDA leave out when looking for features, and why?

*Your answer here.*

## 2. Center, spread and skew

*[OpenIntro Statistics](https://www.openintro.org/book/os/), section 2.1, "Examining numerical data," covers the mean, median, standard deviation, quartiles and skew.*

**Worked example.** `describe()` gives the count, mean, standard deviation, quartiles and range in one table. When the mean is well above the median, a few large values are pulling the mean up and the data is **right-skewed**. For skewed data the median describes a typical value better.

In [ ]:
display(shop["spend_last_month"].describe())
print(shop["spend_last_month"].mean(), shop["spend_last_month"].median())   # mean above median means right skew
print(shop["spend_last_month"].skew())

**Your turn.**

**2a.** Show `describe()` for `visits_last_30_days` and `days_since_last_visit`, and print each column's skew.

In [ ]:
# Your code here


**2b.** For `days_since_last_visit`, compare the mean with the median. What does the gap between them tell you, and which number better describes a typical member?

*Your answer here.*

## 3. Histograms, box plots and outliers

*[OpenIntro Statistics](https://www.openintro.org/book/os/), section 2.1, also covers histograms, box plots and robust statistics.*

**Worked example.** A **histogram** shows the shape of a column. A **box plot** shows the median, the quartiles, and points beyond the **IQR fences**, 1.5 times the interquartile range past the quartiles. Points beyond the fences are worth a look, but being unusual doesn't make a value wrong.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].hist(shop["spend_last_month"], bins=30, color="#2a78d6", edgecolor="white")
axes[0].set(title="Spend last month", xlabel="Dollars", ylabel="Customers")
axes[1].boxplot(shop["spend_last_month"], vert=False)
axes[1].set(title="Spend last month", xlabel="Dollars", yticks=[])
plt.tight_layout()
plt.show()

q1, q3 = shop["spend_last_month"].quantile([0.25, 0.75])
iqr = q3 - q1
outlier = (shop["spend_last_month"] < q1 - 1.5 * iqr) | (shop["spend_last_month"] > q3 + 1.5 * iqr)
print(outlier.sum(), "customers fall outside the IQR fences")

**Your turn.**

**3a.** Plot a histogram of `visits_last_30_days` and of `days_since_last_visit`, side by side, with titles and axis labels. Then use the IQR rule to count the outliers in each column.

In [ ]:
# Your code here


**3b.** Should you delete the outliers in `days_since_last_visit`? Explain how an unusual value differs from an impossible one, like the ones you cleaned in Module 4.

*Your answer here.*

## 4. Category frequencies

*[OpenIntro Statistics](https://www.openintro.org/book/os/), section 2.2, "Considering categorical data," covers frequency tables and bar plots.*

**Worked example.** `value_counts` shows how often each category appears, and `normalize=True` gives shares. A bar chart makes the same counts easy to compare. Watch for rare categories, because rates computed from a handful of rows are unreliable.

In [ ]:
print(shop["favorite"].value_counts())
print(shop["favorite"].value_counts(normalize=True).round(3))

counts = shop["favorite"].value_counts()
plt.bar(counts.index, counts.values, color="#2a78d6")
plt.title("Favorite drink")
plt.ylabel("Customers")
plt.show()

**Your turn.**

**4a.** Show the counts and shares for `plan`, `home_gym`, `autopay` and `support_tickets_90_days`, and draw a bar chart of the `plan` counts.

In [ ]:
# Your code here


**4b.** Which values of `support_tickets_90_days` are rare? Why would a rare category cause trouble when you compare churn rates or train a model, and what could you do about it?

*Your answer here.*

## 5. Numeric features compared with churn

*[OpenIntro Statistics](https://www.openintro.org/book/os/), section 2.2, also compares numerical data across groups.*

**Worked example.** To see whether a numeric feature relates to the target, compare its distribution for each outcome. `groupby` gives the means and medians, and side-by-side box plots show the whole spread.

In [ ]:
display(shop.groupby("lapsed")[["visits_last_month", "spend_last_month"]].agg(["mean", "median"]).round(2))

stayed = shop.loc[~shop["lapsed"], "visits_last_month"]
lapsed = shop.loc[shop["lapsed"], "visits_last_month"]
plt.boxplot([stayed, lapsed], tick_labels=["Stayed", "Lapsed"])
plt.title("Visits last month")
plt.ylabel("Visits")
plt.show()

**Your turn.**

**5a.** Compare the mean and median of `visits_last_30_days`, `days_since_last_visit`, `age` and `support_tickets_90_days` for churned and non-churned members. Then draw side-by-side box plots of `visits_last_30_days` for the two groups.

In [ ]:
# Your code here


**5b.** Which feature differs most between the two groups? Which barely differs at all?

*Your answer here.*

## 6. Categorical features compared with churn

*[OpenIntro Statistics](https://www.openintro.org/book/os/), section 2.2, covers contingency tables and row proportions.*

**Worked example.** `pd.crosstab` with `normalize="index"` gives the share of each row's customers in each outcome, which here is the lapse rate for each category. Show the counts next to the rates, so you can tell a real difference from a small group. A dashed line at the overall rate makes each bar easy to judge.

In [ ]:
rates = pd.crosstab(shop["favorite"], shop["lapsed"], normalize="index")[True]   # share lapsed in each row
counts = shop["favorite"].value_counts()
display(pd.DataFrame({"customers": counts, "lapse_rate": rates.round(3)}))

plt.bar(rates.index, rates.values, color="#2a78d6")
plt.axhline(shop["lapsed"].mean(), color="#52514e", linewidth=1, linestyle="--")   # overall rate for comparison
plt.title("Lapse rate by favorite drink")
plt.ylabel("Share who lapsed")
plt.show()

**Your turn.**

**6a.** Show the churn rate for each value of `plan`, `home_gym` and `autopay`, and draw a bar chart of churn rate by plan with a line at the overall rate.

In [ ]:
# Your code here


**6b.** Which categorical feature shows the biggest difference in churn rate? Is that difference big enough that you'd expect it to help a model?

*Your answer here.*

## 7. Correlation

*[OpenIntro Statistics](https://www.openintro.org/book/os/), section 8.1, "Line fitting, residuals, and correlation," explains correlation.*

**Worked example.** **Correlation** measures how closely two numeric columns move together in a straight line, from -1 to 1. Turning `True`/`False` into 1/0 lets you correlate a feature with the target. Correlation misses relationships that aren't straight lines, so it doesn't replace the plots.

In [ ]:
numeric = shop[["spend_last_month", "visits_last_month", "member", "lapsed"]].astype(float)
display(numeric.corr().round(2))
print(numeric.corr()["lapsed"].drop("lapsed").sort_values())

**Your turn.**

**7a.** Compute the correlation of each numeric feature with `churned` and sort the results. Then compute the correlation between `visits_last_30_days` and `visits_prior_30_days`.

In [ ]:
# Your code here


**7b.** `visits_last_30_days` and `visits_prior_30_days` are strongly correlated with each other. Why might that matter when you build a model?

*Your answer here.*

## 8. Significance compared with effect size

*[OpenIntro Statistics](https://www.openintro.org/book/os/), section 5.3, "Hypothesis testing," explains p-values. Section 7.3 covers the t-test for a difference of two means, and section 6.4 covers the chi-square test for two-way tables.*

**Worked example.** A **t-test** asks whether two groups' means differ by more than chance would explain, and a **chi-square test** asks the same about rates across categories. A small p-value (often below 0.05) means the difference is unlikely to be chance. It doesn't mean the difference is big. The **effect size**, the actual difference in means or in percentage points, tells you whether it matters. With enough rows, even a tiny difference gets a tiny p-value, as the last part of the example shows.

In [ ]:
from scipy import stats

# t-test for a numeric feature: do lapsed and staying customers differ in average visits?
t = stats.ttest_ind(shop.loc[shop["lapsed"], "visits_last_month"],
                    shop.loc[~shop["lapsed"], "visits_last_month"], equal_var=False)
print(f"t = {t.statistic:.2f}, p = {t.pvalue:.2g}")

# chi-square for a categorical feature: is lapsing related to favorite drink?
chi = stats.chi2_contingency(pd.crosstab(shop["favorite"], shop["lapsed"]))
print(f"chi-square = {chi.statistic:.2f}, p = {chi.pvalue:.2g}")

# With enough rows, even a tiny difference is "significant"
big = rng.random(200_000) < 0.30
group = rng.random(200_000) < 0.5
big = big | (group & (rng.random(200_000) < 0.01))   # group adds under 1 percentage point
print(pd.Series(big).groupby(group).mean().round(4).to_dict())
print(stats.chi2_contingency(pd.crosstab(group, big)).pvalue)

**Your turn.**

**8a.** Run t-tests comparing churned and non-churned members on `visits_last_30_days` and on `age`. Run chi-square tests of `churned` against `autopay`, `home_gym` and `plan`. Print each p-value.

In [ ]:
# Your code here


**8b.** Which features have p-values below 0.05? For one feature, explain what the p-value tells you and what it doesn't.

*Your answer here.*

## 9. Warning signs of leakage

*Module 3 introduced leakage. The [scikit-learn page on common pitfalls](https://scikit-learn.org/stable/common_pitfalls.html) covers it under "Data leakage."*

**Worked example.** Leakage often shows up in EDA as a relationship that's too good to be true. In the example, every customer who received a win-back coupon had already lapsed, because the shop mails the coupon *after* a customer lapses. The column predicts the target perfectly and is useless for prediction.

In [ ]:
# The shop mails a win-back coupon to customers after they lapse
shop["winback_coupon_sent"] = shop["lapsed"] & (rng.random(500) < 0.8)
display(pd.crosstab(shop["winback_coupon_sent"], shop["lapsed"], normalize="index").round(3))

**Your turn.**

**9a.** Compare the churn rate for members with a `cancellation_reason` filled in with the rate for everyone else.

In [ ]:
# Your code here


**9b.** The difference is large. Why can't `cancellation_reason` be a feature anyway? Name two warning signs of leakage you could spot during EDA.

*Your answer here.*

## Bring to class

Write down anything that confused you. Tuesday's walk-through covers sections 1–4, Thursday's class covers sections 5–9, and the answer key is posted after Tuesday's class.

*Your answer here.*